In [1]:
# ⬅︎ 第 1 个 Code cell
# !pip install -q ipywidgets ipycanvas nest_asyncio
import nest_asyncio, asyncio, io, cv2, pickle, json, logging
import numpy as np
from pathlib import Path
from ipycanvas import Canvas, hold_canvas
from ipywidgets import VBox, HBox, Button, Label, IntSlider, Dropdown
from IPython.display import display
nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)


In [17]:
%gui asyncio

In [2]:
# ⬅︎ 第 2 个 Code cell
# 1) 数据根目录 —— 即 processed_data/{TASK_NAME}/demo_xxx/
DATA_DIR      = Path("/home/bobby/data/processed_data")

# 2) 要标注的任务
TASK_NAME     = "place_bottle_from_the_fridge_left_robot"  # e.g. "pick_bottle_from_the_fridge_left_robot"

# 3) 两个相机的索引（必须与 convert_to_pkl_robot.py 相同）
# CAMERA_IDS    = [2, 5]
CAMERA_IDS    = [4, 6]  # e.g. [2, 5] for left and right cameras

# 4) 裁剪与缩放参数 —— 与 convert_to_pkl_robot.py 完全一致
CROP_H, CROP_W  = (0.0, 1.0), (0.0, 1.0)
SAVE_IMG_SIZE   = (256, 256)

# 5) 保存目录：coordinates/{TASK_NAME}/{demo_name}.pkl
OUT_DIR = Path("coordinates") / TASK_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("准备写入到：", OUT_DIR.resolve())


准备写入到： /home/bobby/Point-Policy/point_policy/robot_utils/franka/coordinates/place_bottle_from_the_fridge_left_robot


In [3]:
# ⬅︎ 第 3 个 Code cell
def read_last_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open {video_path}")

    last = None
    while True:
        ret, frame = cap.read()
        if not ret:      # 到文件结尾
            break
        last = frame     # 只有读取成功才更新

    cap.release()
    if last is None:
        raise IOError(f"No frames found in {video_path}")
    return last         # BGR


def crop_resize(frame):
    h, w = frame.shape[:2]
    frame = frame[int(h*CROP_H[0]):int(h*CROP_H[1]),
                  int(w*CROP_W[0]):int(w*CROP_W[1])]
    return cv2.resize(frame, SAVE_IMG_SIZE)

class ClickCanvas:
    """单张图的点击工具，点击一次即记录 (x,y)。"""
    def __init__(self, rgb, title):
        self.coord = None
        h, w = rgb.shape[:2]
        self.canvas = Canvas(width=w, height=h)
        self.canvas.put_image_data(rgb, 0, 0)
        self.label  = Label(value=f"{title}: 未选择")
        self.title  = title

        def _on_click(x, y):
            self.coord = (x, y)
            self.label.value = f"{title}: ({x}, {y})"
            with hold_canvas(self.canvas):
                self.canvas.put_image_data(rgb, 0, 0)
                self.canvas.fill_style = "red"
                self.canvas.fill_circle(x, y, 3)

        self.canvas.on_mouse_down(_on_click)


In [4]:
# ⬅︎ 修正版：第 4 个 Code cell
async def annotate_one_demo(demo_path):        # ← 加上 async
    # 读取两台相机的最后一帧
    imgs = []
    for cam in CAMERA_IDS:
        vid = demo_path / "videos" / f"camera{cam}.mp4"
        frame = read_last_frame(vid)
        if frame is None:
            raise RuntimeError(f"{vid} 读取失败")
        imgs.append(crop_resize(frame)[:, :, ::-1])  # 转 RGB

    canvases = [ClickCanvas(imgs[i], f"cam_{CAMERA_IDS[i]}") for i in range(2)]
    done_btn = Button(description="✅ 保存并进入下一条示范")
    hbox = HBox([VBox([c.canvas, c.label]) for c in canvases])
    ui = VBox([hbox, done_btn])
    display(ui)

    fut = asyncio.get_event_loop().create_future()
    def _done(_):
        fut.set_result(None)
    done_btn.on_click(_done)
    await fut
    ui.close()

    # 返回坐标
    return {f"cam_{CAMERA_IDS[i]}": canvases[i].coord for i in range(2)}


In [5]:
# ⬅︎ 第 5 个 Code cell
# async def run_all():
#     demos = sorted([p for p in (DATA_DIR/TASK_NAME).iterdir() if p.is_dir()])
#     for demo in demos:
#         out_file = OUT_DIR / f"{demo.name}.pkl"
#         if out_file.exists():
#             logging.info(f"跳过已标注 demo: {demo.name}")
#             continue
#         logging.info(f"=== 标注 {demo.name} ===")
#         coords = await annotate_one_demo(demo)
#         with open(out_file, "wb") as f:
#             pickle.dump(coords, f)
#         logging.info(f"已保存 → {out_file.name}")

# asyncio.run(run_all())
# ⬅︎ 新版 第 5 个 Code cell
demos = sorted([p for p in (DATA_DIR / TASK_NAME).iterdir() if p.is_dir()])

async def run_all():
    for demo in demos:
        out_file = OUT_DIR / f"{demo.name}.pkl"
        if out_file.exists():
            logging.info(f"跳过已标注 demo: {demo.name}")
            continue
        logging.info(f"=== 标注 {demo.name} ===")
        coords = await annotate_one_demo(demo)
        with open(out_file, "wb") as f:
            pickle.dump(coords, f)
        logging.info(f"已保存 → {out_file.name}")

await run_all()        # ← 直接 await，而不是 asyncio.run()



INFO:root:=== 标注 demonstration_1 ===


CancelledError: 

In [1]:
# # 安装依赖
# !pip install -q ipywidgets ipycanvas nest_asyncio

import nest_asyncio, asyncio, io, cv2, pickle, logging, json
import numpy as np
from pathlib import Path
from ipycanvas import Canvas, hold_canvas
from ipywidgets import VBox, HBox, Button, Label
from IPython.display import display
nest_asyncio.apply()
logging.basicConfig(level=logging.INFO)

# ==== 全局配置 ====
DATA_DIR   = Path("/home/bobby/data/processed_data")   # 你的 processed_data 路径
TASK_NAME  = "place_bottle_from_the_fridge_left_robot"  # 要标注的任务名称
# TASK_NAME  = "test2"  # e.g. "pick_bottle_from_the_fridge_left_robot"
# CAM_IDS    = [2, 5]              # 两台相机，与 convert 脚本保持一致
CAM_IDS    = [4, 6]              # 两台相机，与 convert 脚本保持一致
CROP_H, CROP_W = (0.0, 1.0), (0.0, 1.0)
SAVE_SZ         = (256, 256)     # 与 convert 中的 save_img_size 相同
OUT_DIR  = Path("coordinates") / TASK_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ==== 工具函数 ====
def read_last_frame(mp4):
    cap = cv2.VideoCapture(str(mp4))
    if not cap.isOpened():
        raise IOError(f"Cannot open {mp4}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total-1)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise IOError(f"Cannot read last frame from {mp4}")
    return frame                       # BGR

def crop_resize(bgr):
    h, w = bgr.shape[:2]
    bgr = bgr[int(h*CROP_H[0]):int(h*CROP_H[1]),
              int(w*CROP_W[0]):int(w*CROP_W[1])]
    return cv2.resize(bgr, SAVE_SZ)

class ClickOnce:
    """单张图：点击一次记录坐标"""
    def __init__(self, rgb, title):
        self.coord = None
        h, w = rgb.shape[:2]
        self.canvas = Canvas(width=w, height=h)
        self.canvas.put_image_data(rgb, 0, 0)
        self.label  = Label(value=f"{title}: 未选择")

        def _on_click(x, y):
            self.coord = (x, y)
            self.label.value = f"{title}: ({x}, {y})"
            with hold_canvas(self.canvas):
                self.canvas.put_image_data(rgb, 0, 0)
                self.canvas.fill_style = "red"
                self.canvas.fill_circle(x, y, 5)

        self.canvas.on_mouse_down(_on_click)


In [15]:
# -------- Cell 2：每运行一次 = 标注一条示范 --------
import re, os, pickle, logging
from ipywidgets import Button, VBox, HBox
from IPython.display import display

def numeric_key(path):
    """确保 demonstration_2 < demonstration_10"""
    m = re.search(r'(\d+)$', path.name)
    return int(m.group(1)) if m else path.name

def coords_valid(pkl_path):
    """文件存在且内容包含有效坐标"""
    if not pkl_path.exists():
        return False
    try:
        data = pickle.load(open(pkl_path, 'rb'))
        # return ("cam_2" in data and "cam_5" in data
        #         and data["cam_2"] is not None
        #         and data["cam_5"] is not None)
        return ("cam_4" in data and "cam_6" in data
                and data["cam_4"] is not None
                and data["cam_6"] is not None)
    except Exception:
        return False

# 1) 找到下一条需要标注的 demo
all_demos = sorted(
    [p for p in (DATA_DIR / TASK_NAME).iterdir() if p.is_dir()],
    key=numeric_key
)
demo = None
for d in all_demos:
    if not coords_valid(OUT_DIR / f"{d.name}.pkl"):
        demo = d
        break

if demo is None:
    print("🎉 所有示范都已标注完毕！")
    raise SystemExit

print(f"👉 正在标注: {demo.name}")

# 2) 读取两台相机最后一帧
imgs = []
for cam in CAM_IDS:
    mp4 = demo / "videos" / f"camera{cam}.mp4"
    bgr = read_last_frame(mp4)
    rgb = crop_resize(bgr)[:, :, ::-1]
    imgs.append(rgb)

# 3) 可视化并点击
canvases = [ClickOnce(imgs[i], f"cam_{CAM_IDS[i]}") for i in range(2)]
save_btn  = Button(description="💾 Save & Next")
ui        = VBox([HBox([VBox([c.canvas, c.label]) for c in canvases]), save_btn])
display(ui)

def _save(btn):
    # a. 检查是否两幅图都已点击
    if any(c.coord is None for c in canvases):
        for c in canvases:
            if c.coord is None:
                c.label.value = "❗ 请先点击这里"
        return
    
    # b. 写入 pkl（覆盖旧文件）
    coords   = {f"cam_{CAM_IDS[i]}": canvases[i].coord for i in range(2)}
    out_path = OUT_DIR / f"{demo.name}.pkl"
    with open(out_path, "wb") as f:
        pickle.dump(coords, f)
    print(f"✅ 已保存 → {out_path.name}")
    
    # c. 关闭 UI，并提示手动继续
    ui.close()
    print("👉 重新运行此单元 (Run / Shift-Enter) 进入下一条示范")

save_btn.on_click(_save)


🎉 所有示范都已标注完毕！


SystemExit: 

/home/bobby/miniconda3/envs/point-policy/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
